#IMDB Dataset

In [ ]:
vocab_size=10000

from tensorflow.keras.datasets import imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=vocab_size)

17464789/17464789 [==============================] - 1s 0us/step


In [ ]:
word_index = imdb.get_word_index()
reverse_word_index = dict(
    [(value, key) for (key, value) in word_index.items()])

1641221/1641221 [==============================] - 1s 1us/step


In [ ]:
for doc_index in range(0,10):
  decoded_review = " ".join([reverse_word_index.get(i - 3, "?") for i in train_data[doc_index]])
  print(f'[{doc_index}]: {decoded_review}')

[0]: ? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't yo

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, Flatten, Dense

vocab_size = 10000
max_length = 50
embedding_dim = 8

# Assuming train_data, train_labels, test_data, and test_labels are already defined
train_data = pad_sequences(train_data, maxlen=max_length)
test_data = pad_sequences(test_data, maxlen=max_length)

# Configurations for CNN layers
filters = [16, 32]
kernel_sizes = [5, 7]
use_maxpool = [False, True]
conv_layers = [1, 2]

# Function to build and evaluate the model
def build_and_evaluate_model(conv_layers, filter_size, kernel_size, pool):
    model = Sequential()
    model.add(Embedding(vocab_size, embedding_dim, input_length=max_length))

    for _ in range(conv_layers):
        model.add(Conv1D(filter_size, kernel_size, activation='relu'))
        if pool:
            model.add(MaxPooling1D(pool_size=2))

    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))

    print(f"Model Architecture with {conv_layers} Conv layers, filter size {filter_size}, kernel size {kernel_size}, maxpool {pool}:")
    model.summary()

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    history = model.fit(train_data, train_labels,
                        epochs=5,
                        batch_size=64,
                        validation_data=(test_data, test_labels))

    y_pred_prob = model.predict(test_data)
    y_pred = np.round(y_pred_prob).astype(int)

    cm = confusion_matrix(test_labels, y_pred)
    print(f"Confusion Matrix for {conv_layers} Conv layers, filter size {filter_size}, kernel size {kernel_size}, maxpool {pool}:")
    print(cm)

    report = classification_report(test_labels, y_pred)
    print(f"Classification Report for {conv_layers} Conv layers, filter size {filter_size}, kernel size {kernel_size}, maxpool {pool}:")
    print(report)

    print(f"Final Training Accuracy for {conv_layers} Conv layers, filter size {filter_size}, kernel size {kernel_size}, maxpool {pool}: {history.history['accuracy'][-1]}")
    print(f"Final Validation Accuracy for {conv_layers} Conv layers, filter size {filter_size}, kernel size {kernel_size}, maxpool {pool}: {history.history['val_accuracy'][-1]}")

# Loop through all combinations
for conv_layer in conv_layers:
    for filter_size in filters:
        for kernel_size in kernel_sizes:
            for pool in use_maxpool:
                build_and_evaluate_model(conv_layer, filter_size, kernel_size, pool)

Model Architecture with 1 Conv layers, filter size 16, kernel size 5, maxpool False:
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 50, 8)             80000     
                                                                 
 conv1d (Conv1D)             (None, 46, 16)            656       
                                                                 
 flatten (Flatten)           (None, 736)               0         
                                                                 
 dense (Dense)               (None, 1)                 737       
                                                                 
Total params: 81393 (317.94 KB)
Trainable params: 81393 (317.94 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/5
391/391 [==============================] - 27s 58ms/ste

#Persian dataset

In [1]:
!pip install datasets
from datasets import load_dataset

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.31.0, but you have requests 2.32.3 which is incompatible.


In [3]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, Flatten, Dense
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from datasets import load_dataset

persian_dataset = load_dataset("SeyedAli/Persian-Text-Sentiment")

train_texts = [example['text'] for example in persian_dataset['train']]
train_labels = [example['label'] for example in persian_dataset['train']]
test_texts = [example['text'] for example in persian_dataset['test']]
test_labels = [example['label'] for example in persian_dataset['test']]

vocab_size = 10000
max_length = 50
embedding_dim = 8

train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(train_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

train_data = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
test_data = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')

def build_and_evaluate_model(conv_layers, filter_size, kernel_size, pool):
    model = Sequential()
    model.add(Embedding(vocab_size, embedding_dim, input_length=max_length))

    for _ in range(conv_layers):
        model.add(Conv1D(filter_size, kernel_size, activation='relu'))
        if pool:
            model.add(MaxPooling1D(pool_size=2))

    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))

    print(f"Model Architecture with {conv_layers} Conv layers, filter size {filter_size}, kernel size {kernel_size}, maxpool {pool}:")
    model.summary()

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    model.fit(train_data, train_labels,
              epochs=5,
              batch_size=64,
              validation_data=(test_data, test_labels))

    accuracy = evaluate_model(model, test_data, test_labels)
    print(f"Final Accuracy for {conv_layers} Conv layers, filter size {filter_size}, kernel size {kernel_size}, maxpool {pool}: {accuracy}\n")
    return accuracy

def evaluate_model(model, test_data, test_labels):
    predictions = model.predict(test_data)
    predictions_binary = [1 if p > 0.5 else 0 for p in predictions]

    accuracy = accuracy_score(test_labels, predictions_binary)
    precision = precision_score(test_labels, predictions_binary)
    recall = recall_score(test_labels, predictions_binary)
    f1 = f1_score(test_labels, predictions_binary)

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)

    cm = confusion_matrix(test_labels, predictions_binary)
    print("Confusion Matrix:")
    print(cm)

    return accuracy

filters = [16, 32]
kernel_sizes = [5, 7]
use_maxpool = [False, True]
conv_layers = [1, 2]

results = []
for conv_layer in conv_layers:
    for filter_size in filters:
        for kernel_size in kernel_sizes:
            for pool in use_maxpool:
                accuracy = build_and_evaluate_model(conv_layer, filter_size, kernel_size, pool)
                results.append((conv_layer, filter_size, kernel_size, pool, accuracy))

for result in results:
    conv_layer, filter_size, kernel_size, pool, accuracy = result
    print(f"Conv Layer = {conv_layer} , Filter Size = {filter_size}, Kernel Size = {kernel_size}, MaxPooling = {'Yes' if pool else 'No'} : Final Accuracy = {accuracy}")

Model Architecture with 1 Conv layers, filter size 16, kernel size 5, maxpool False:
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 50, 8)             80000     
                                                                 
 conv1d (Conv1D)             (None, 46, 16)            656       
                                                                 
 flatten (Flatten)           (None, 736)               0         
                                                                 
 dense (Dense)               (None, 1)                 737       
                                                                 
Total params: 81393 (317.94 KB)
Trainable params: 81393 (317.94 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/5
873/873 [==============================] - 39s 40ms/ste

#Third Dataset

In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, Flatten, Dense
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import numpy as np

dataset = load_dataset("saied/persian_news_dataset", split="train", streaming=True)

docs = []
document_count = 1000
counter = 0
skip_counter = 0

for doc in dataset:
    if len(doc['title']) == 0 or len(doc['category']) == 0:
        skip_counter += 1
        if skip_counter % 10000 == 0:
            print(f'{skip_counter} skipped')
        continue
    docs.append(doc)
    counter += 1
    if counter == document_count:
        break

texts = [doc['text'] for doc in docs]
categories = [doc['category'] for doc in docs]

max_words = 10000
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

input_length = 50
X = pad_sequences(sequences, maxlen=input_length)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(categories)
number_of_classes = len(label_encoder.classes_)
y = tf.keras.utils.to_categorical(y, num_classes=number_of_classes)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

embedding_dim = 100

def build_and_evaluate_model(conv_layers, filter_size, kernel_size, pool):
    model = Sequential()
    model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_length))

    for _ in range(conv_layers):
        model.add(Conv1D(filter_size, kernel_size, activation='relu'))
        if pool:
            model.add(MaxPooling1D(pool_size=2))

    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(number_of_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_val, y_val), verbose=2)

    val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
    y_pred = np.argmax(model.predict(X_val), axis=1)
    y_true = np.argmax(y_val, axis=1)

    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)

    print(f"Accuracy: {val_accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 Score: {f1}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Final Accuracy for {conv_layers} Conv layers, filter size {filter_size}, kernel size {kernel_size}, maxpool {pool}: {val_accuracy}\n")

    return val_accuracy

filters = [16, 32]
kernel_sizes = [5, 7]
use_maxpool = [False, True]
conv_layers = [1, 2]

results = []
for conv_layer in conv_layers:
    for filter_size in filters:
        for kernel_size in kernel_sizes:
            for pool in use_maxpool:
                accuracy = build_and_evaluate_model(conv_layer, filter_size, kernel_size, pool)
                results.append((conv_layer, filter_size, kernel_size, pool, accuracy))

for result in results:
    conv_layer, filter_size, kernel_size, pool, accuracy = result
    print(f"Conv Layer = {conv_layer} , Filter Size = {filter_size}, Kernel Size = {kernel_size}, MaxPooling = {'Yes' if pool else 'No'} : Final Accuracy = {accuracy}")

/usr/local/lib/python3.10/dist-packages/datasets/load.py:1491: FutureWarning: The repository for saied/persian_news_dataset contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/saied/persian_news_dataset
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


10000 skipped
20000 skipped
30000 skipped
40000 skipped
50000 skipped
60000 skipped
70000 skipped
80000 skipped
90000 skipped
100000 skipped
110000 skipped
120000 skipped
130000 skipped
140000 skipped
150000 skipped
160000 skipped
170000 skipped
180000 skipped
190000 skipped
200000 skipped
210000 skipped
Epoch 1/5
25/25 - 7s - loss: 1.6464 - accuracy: 0.2862 - val_loss: 1.5477 - val_accuracy: 0.3850 - 7s/epoch - 266ms/step
Epoch 2/5
25/25 - 2s - loss: 1.1150 - accuracy: 0.6950 - val_loss: 0.8714 - val_accuracy: 0.7650 - 2s/epoch - 82ms/step
Epoch 3/5
25/25 - 2s - loss: 0.3201 - accuracy: 0.9488 - val_loss: 0.2040 - val_accuracy: 0.9750 - 2s/epoch - 79ms/step
Epoch 4/5
25/25 - 1s - loss: 0.0319 - accuracy: 1.0000 - val_loss: 0.0780 - val_accuracy: 0.9750 - 916ms/epoch - 37ms/step
Epoch 5/5
25/25 - 1s - loss: 0.0068 - accuracy: 1.0000 - val_loss: 0.0557 - val_accuracy: 0.9850 - 1s/epoch - 42ms/step
7/7 [==============================] - 0s 2ms/step
Accuracy: 0.9850000143051147
Precision: